# 🔥 Advanced · ControlNet

> 🔥 **Advanced / heavy lab.** Generate images that follow a structure map (Canny edges, pose, depth) with ControlNet.
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **10–20 min** including downloads. Built on the official **[diffusers ControlNet](https://huggingface.co/docs/diffusers/using-diffusers/controlnet)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

Conditional control over a diffusion foundation model — links to track B (geometry/structure) and to pose (track A).

In [ ]:
!nvidia-smi -L
!pip install -q diffusers accelerate transformers opencv-python controlnet_aux

## 1 · Build a control image (Canny edges of your input)

In [ ]:
import cv2, numpy as np
from PIL import Image
img = np.array(Image.open("input.jpg").convert("RGB"))
edges = cv2.Canny(img, 100, 200)
control = Image.fromarray(np.stack([edges] * 3, -1))
control.save("control.png")

## 2 · Generate following that structure

In [ ]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
cn = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", controlnet=cn, torch_dtype=torch.float16).to("cuda")
out = pipe("a photorealistic robot, studio lighting", image=control, num_inference_steps=30).images[0]
out.save("controlled.png"); out

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`.

## How this links to tracks A–D
Structural control links to geometry:
- **A · Human** pose-conditioned generation · **B · 3D** depth / edge / normal control.

## Next steps
- Swap the conditioner for **OpenPose** (links to the pose lab, track A) or **depth/normal** maps (track B).
- Train your own ControlNet to condition on a new modality; combine with the LoRA lab.